# Compound Event Asset Prioritization

In this notebook, we'll be identifying incidents where high winds co-occur with extreme multi-day precipitation events over the Southern California Edison (SCE) service territory. We will also overlay transmission line infrastructure
to highlight which grid assets are most exposed.

**Intended Application:** As a user, I want to **<span style="color:red">identify what areas may be most exposed to compound high-wind and extreme-precipitation events, and how that exposure changes between GWL 0.8°C and 2.0°C.</span>**

**Runtime:** ~30 min. to run through the whole notebook. Modifications to different queries may increase/decrease the overall runtime.

### Real-World Motivation: Atmospheric River from February 3–5, 2024 across California

A classic atmospheric river fueled by El Niño produced historic rainfall across California during
this event. At its peak, the Weather Prediction Center issued an unusual **"high risk of excessive
rainfall"** alert covering more than 16 million people across Southern California. Strong winds
knocked out power to over 300,000 customers, and three deaths were reported from fallen trees as
hundreds of mudslides damaged homes, businesses, and roadways.

This event exemplifies the compound wind-and-precipitation hazard analyzed in this notebook:

| Sub-condition | Indicator | Threshold |
|---|---|---|
| Strong winds | Daily max wind speed within SCE Service Territory | > 20 mph |
| Intense precipitation | 3-day rolling precipitation total within SCE Service Territory | > 3 inches |

#### Imports

In [ ]:
import os
import sys

import climakitae as ck
import dask
import geopandas as gpd
import xarray as xr
from climakitae.core.data_export import export
from climakitae.new_core.data_access import DataCatalog
from dask.diagnostics import ProgressBar

sys.path.append(os.path.abspath('.'))
from wg10_helpers import (plot_compound_event_maps,
                                plot_metric_frequency_maps)

cd = ck.ClimateData(verbosity=-1)

## User Selections
We will specify the following parameters for this Compound Event Asset Prioritization example.

In [ ]:
GWLS = [0.8, 2.0]
territory = "Southern California Edison"
resolution = "d03"                    # 3 km
precip_threshold_inches  = 3.0        # 3-day rolling total threshold (inches)
wind_speed_threshold_mph = 20.0       # daily max wind speed threshold (mph)

# Map extent for all figures
LON_EXTENT = (-121.0, -114.0)
LAT_EXTENT = (33.0, 38.5)

## Retrieve Data

In [ ]:
cd.reset() # reset any previous query settings to avoid unintended carryover
procs = {
    "warming_level": {
        "warming_levels": GWLS,
        "warming_level_window": 15,   # ±15 yr around GWL crossing = 30 yr sample per sim
    },
    "clip": territory,
    "filter_unadjusted_models": "yes",  # retain only bias-adjusted simulations
}

In [ ]:
# Retrieving precip WRF data
procs['convert_units']  = 'inches/d'
variable = 'prec'
precip_data = (
    cd.catalog("cadcat")
        .activity_id("WRF")
        .institution_id("UCLA")
        .table_id("day")
        .grid_label(resolution)
        .variable(variable)
        .processes(procs)
        .get()
    )

# Retrieving wind speed WRF data
procs['convert_units']  = 'mph'
variable = 'wspd10max'
wind_speed_data = (
    cd.catalog("cadcat")
        .activity_id("WRF")
        .institution_id("UCLA")
        .table_id("day")
        .grid_label(resolution)
        .variable(variable)
        .processes(procs)
        .get()
    )

In [ ]:
# Sanity check: confirm expected dims {warming_level:2, sim:5, time_delta:10950, y:240, x:156}
print("Precip dims:", dict(precip_data.dims))
print("Wind speed dims:", dict(wind_speed_data.dims))

## Compute Spatial Mask

Derives invalid gridcells mask from the first time step of precipitation data.

In [ ]:
# Grabbing a spatial mask needed for dropping gridcells outside of area of interest
mask = (~precip_data.isel(warming_level=0, time_delta=0, sim=0).prec.isnull()).compute()

## Load Boundaries and Transmission Lines

Loads SCE service territory boundary and clipped transmission lines.

In [ ]:
TX_CACHE = "data/tx_lines_clipped_cache.parquet"
tx_lines_clipped = gpd.read_parquet(TX_CACHE)
sce_boundary = DataCatalog().boundaries._ca_utilities.query("Utility == 'Southern California Edison'")

## Count Compound Events

For each GWL, `count_compound_events` computes:
- 3-day rolling precipitation totals
- Compound flag: rolling precip > threshold AND wind > threshold on ANY 1 of those 3 days

Returns four metrics per simulation:
- `compound_freq`: avg compound events per grid cell per year
- `wind_freq`: avg days with max wind speed above threshold per year
- `precip_freq`: avg 3-day rolling windows above precip threshold per year
- `compound_count`: number of grid cells meeting compound conditions per day (timeseries)

In [ ]:
def count_compound_events(precip_da, wind_da, wind_threshold, duration=3,
                           precip_thresh=10.0, n_years=30.0):
    """
    Compute compound event frequency and sub-condition metrics per grid cell.

    A compound event day is one where:
      - the `duration`-day backward rolling precipitation sum exceeds `precip_thresh`, AND
      - wind speed exceeded `wind_threshold` on at least one of those same `duration` days.

    Parameters
    ----------
    precip_da : xr.DataArray
        Daily precipitation (inches/day), dims (sim, time_delta, y, x).
    wind_da : xr.DataArray
        Daily max wind speed (mph), dims (sim, time_delta, y, x).
    wind_threshold : float
        Wind speed threshold (mph) for the wind sub-condition.
    duration : int
        Rolling window length in days. Default 3.
    precip_thresh : float
        Rolling precipitation total threshold (inches). Default 3.0.
    n_years : float
        Number of years in the sample window, used to convert counts to annual rates.

    Returns
    -------
    dict of lazy xr.DataArrays:
        compound_freq  : (sim, y, x)       avg compound events / yr
        wind_freq      : (sim, y, x)       avg days wind > threshold / yr
        precip_freq    : (sim, y, x)       avg rolling windows > precip_thresh / yr
        compound_count : (sim, time_delta) grid cells meeting compound conditions per day
    """
    # Rolling precipitation sum — requires all `duration` days present (min_periods=duration)
    rolling_precip = precip_da.rolling(time_delta=duration, min_periods=duration).sum()

    # Wind hit: True on days wind exceeds threshold
    wind_hit = wind_da > wind_threshold
    # wind_any: True on day N if wind hit on any of the previous `duration` days
    wind_any = wind_hit.rolling(time_delta=duration, min_periods=1).max()

    precip_hit = rolling_precip > precip_thresh
    compound   = precip_hit & (wind_any == 1)

    return {
        "compound_freq":  compound.sum(dim="time_delta") / n_years,
        "wind_freq":      wind_hit.sum(dim="time_delta") / n_years,
        "precip_freq":    precip_hit.sum(dim="time_delta") / n_years,
        "compound_count": compound.sum(dim=["y", "x"]),
    }

In [ ]:
### Computing the compound event metrics is the most computationally intensive step, 
### so we build lazy task graphs with dask and monitor progress with ProgressBar.

def get_lazy(precip_da, wind_da, wind_threshold, precip_thresh):
    """Return (keys, lazy_values) for count_compound_events without triggering compute."""
    lazy = count_compound_events(precip_da, wind_da, wind_threshold, precip_thresh=precip_thresh)
    return list(lazy.keys()), list(lazy.values())


# Build lazy task graphs for each GWL separately. No computation happens here —
# dask defers execution until dask.compute() is called below.
keys_b, vals_b = get_lazy(
    precip_data.sel(warming_level=0.8)["prec"],
    wind_speed_data.sel(warming_level=0.8)["wspd10max"],
    wind_speed_threshold_mph,
    precip_thresh=precip_threshold_inches,
)

keys_f, vals_f = get_lazy(
    precip_data.sel(warming_level=2.0)["prec"],
    wind_speed_data.sel(warming_level=2.0)["wspd10max"],
    wind_speed_threshold_mph,
    precip_thresh=precip_threshold_inches,
)

# Two separate dask.compute calls so the ProgressBar can report each GWL independently.
# Combining both into one call would be slightly faster but harder to monitor.
with ProgressBar():
    print("Computing baseline GWL results...")
    all_computed_b = dask.compute(*vals_b)
    
with ProgressBar():
    print("Computing future GWL results...")
    all_computed_f = dask.compute(*vals_f)

# Repack flat result tuples back into named dicts and store under a common key
res_baseline   = dict(zip(keys_b, all_computed_b))
res_future     = dict(zip(keys_f, all_computed_f))
metric_results = {"gwl08": res_baseline, "gwl20": res_future}

## Apply Mask and Save Frequency Maps

Applies the spatial mask and exports 7 netCDF files:
- `compound_events_gwl08.nc`, `compound_events_gwl20.nc`, `compound_events_delta.nc`
- `wind_freq_gwl08.nc`, `wind_freq_gwl20.nc`
- `precip_freq_gwl08.nc`, `precip_freq_gwl20.nc`

Each file has a `sim` dimension (5 simulations). Figures use the median across sims.

In [ ]:
# Apply spatial mask to zero out grid cells outside the SCE territory
baseline_map = metric_results["gwl08"]["compound_freq"].where(mask)
future_map   = metric_results["gwl20"]["compound_freq"].where(mask)
delta_map    = (future_map - baseline_map).where(mask)

print(f"Baseline — max: {float(baseline_map.median('sim').max()):.4f} events/yr")
print(f"Future   — max: {float(future_map.median('sim').max()):.4f} events/yr")
print(f"Delta    — range: [{float(delta_map.median('sim').min()):.4f}, {float(delta_map.median('sim').max()):.4f}]")

# Export compound event maps (both GWLs + delta)
export(baseline_map, f'data/compound_events_gwl08')
export(future_map, f'data/compound_events_gwl20')
export(delta_map, f'data/compound_events_delta')

# Export sub-condition frequency maps (used in Figures 2 and 3)
export(metric_results["gwl08"]["wind_freq"].where(mask), f'data/wind_freq_gwl08')
export(metric_results["gwl20"]["wind_freq"].where(mask), f'data/wind_freq_gwl20')
export(metric_results["gwl08"]["precip_freq"].where(mask), f'data/precip_freq_gwl08')
export(metric_results["gwl20"]["precip_freq"].where(mask), f'data/precip_freq_gwl20')

Now that we've computed the compound event frequency across the SCE service territory for GWLs 0.8 and 2.0, we can move on below to visualizing our results.

---

# Figures

In [ ]:
# Read in saved event files from above
baseline_med   = xr.open_dataarray("data/compound_events_gwl08.nc").median(dim="sim") 
future_med     = xr.open_dataarray("data/compound_events_gwl20.nc").median(dim="sim") 
delta_med      = xr.open_dataarray("data/compound_events_delta.nc").median(dim="sim") 
wind_freq_08   = xr.open_dataarray("data/wind_freq_gwl08.nc").median(dim="sim") 
wind_freq_20   = xr.open_dataarray("data/wind_freq_gwl20.nc").median(dim="sim") 
precip_freq_08 = xr.open_dataarray("data/precip_freq_gwl08.nc").median(dim="sim") 
precip_freq_20 = xr.open_dataarray("data/precip_freq_gwl20.nc").median(dim="sim") 

In [ ]:
# Figure 1: Wind frequency maps
plot_metric_frequency_maps(
    wind_freq_08, wind_freq_20,
    tx_lines=tx_lines_clipped,
    sce_boundary=sce_boundary,
    metric_label=f"Days with Max Wind Speed > {wind_speed_threshold_mph:.0f} mph",
    cmap="Blues",
    unit_label="Days / year",
    out_path="figures/wind_freq_SCE.png",
    lon_extent=LON_EXTENT,
    lat_extent=LAT_EXTENT,
)

In [ ]:
# Figure 2: Precipitation frequency maps
plot_metric_frequency_maps(
    precip_freq_08, precip_freq_20,
    tx_lines=tx_lines_clipped,
    sce_boundary=sce_boundary,
    metric_label=f"3-Day Periods with Precip > {precip_threshold_inches:.0f} in",
    cmap="YlGnBu",
    unit_label="Events / year",
    out_path="figures/precip_freq_SCE.png",
    lon_extent=LON_EXTENT,
    lat_extent=LAT_EXTENT,
)

In [ ]:
# Figure 3: 3-panel compound event frequency map with transmission lines
plot_compound_event_maps(
    baseline_med, future_med, delta_med,
    tx_lines=tx_lines_clipped,
    sce_boundary=sce_boundary,
    out_path="figures/compound_event_asset_prioritization_SCE.png",
    lon_extent=LON_EXTENT,
    lat_extent=LAT_EXTENT,
    title="Compound Event Frequency: High Winds During Extreme 3-Day Precipitation",
)